# 99_e2e_validation.ipynb
**DyPriZa – Ejecución End-to-End (00 → 06) y verificación de artefactos**

Este notebook ejecuta, en orden, los notebooks del pipeline y valida que se generen los archivos esperados.

## 🧭 Orden de ejecución recomendado
0) `00_setup.ipynb` (opcional si tu entorno ya está listo)
1) `01_cleaning_and_eda.ipynb` (opcional; exploratorio)
1b) `01b_occupancy_revpar.ipynb` (opcional)
2) `02_clean_market_data.ipynb`
4) `04_features.ipynb`
5) `05_ml_pricing_basico.ipynb`
5b) `05b_model_testing.ipynb`
6) `06_predictions_consolidation.ipynb`

👉 Este runner ejecuta por defecto: **2, 4, 5, 5b y 6**.


## Parámetros
Puedes activar/desactivar notebooks y fijar tiempo máximo por cada uno.

In [ ]:
from pathlib import Path 


NOTEBOOKS =[
'02_clean_market_data.ipynb',
'04_features.ipynb',
'05_ml_pricing_basico.ipynb',
'05b_model_testing.ipynb',
'06_predictions_consolidation.ipynb',
]


TIMEOUT_PER_NB =1200 


EXPECTED ={
'02_clean_market_data.ipynb':[('data/DyPriZa_MVP.csv',100 )],
'04_features.ipynb':[('data/DyPriZa_features.csv',100 )],
'05_ml_pricing_basico.ipynb':[('data/DyPriZa_predictions.csv',50 ),('models/DyPriZa_best_model.joblib',10 )],
'05b_model_testing.ipynb':[
('data/DyPriZa_cv_results.csv',10 ),
('data/DyPriZa_test_results.csv',10 ),
('data/DyPriZa_rf_feature_importance.csv',10 ),
('data/DyPriZa_permutation_importance.csv',10 ),
('data/DyPriZa_predictions_test_05b.csv',10 ),
('models/DyPriZa_rf_tuned.joblib',10 ),
],
'06_predictions_consolidation.ipynb':[
('data/DyPriZa_predictions_final.csv',10 ),
('data/DyPriZa_predictions_summary.csv',10 ),

],
}

list (EXPECTED .items ())[:2 ]


## 1) Ejecución programática de notebooks con nbclient
Se ejecuta cada notebook en un kernel limpio. Los notebooks ejecutados se guardan con sufijo `_executed.ipynb`.

In [ ]:
import nbformat 
from nbclient import NotebookClient 
from nbclient .exceptions import CellExecutionError 
from datetime import datetime 

results =[]

def run_notebook (path ):
    print (f"\n▶ Ejecutando: {path}")
    nb =nbformat .read (path ,as_version =4 )
    client =NotebookClient (nb ,timeout =TIMEOUT_PER_NB ,kernel_name ='python3')
    started =datetime .now ()
    try :
        client .execute ()
        dur =(datetime .now ()-started ).total_seconds ()
        out_path =Path (path ).with_name (Path (path ).stem +'_executed.ipynb')
        nbformat .write (nb ,out_path )
        print (f"✅ OK: {path} ({dur:.1f}s). Guardado: {out_path}")
        return True 
    except CellExecutionError as e :
        print (f"❌ Error ejecutando {path}:\n{e}")
        return False 

for nb in NOTEBOOKS :
    ok =run_notebook (nb )
    results .append ((nb ,ok ))

results 


## 2) Verificación de artefactos generados
Se comprueba existencia y tamaño mínimo (bytes) de cada archivo esperado por notebook.

In [ ]:
import os 
import pandas as pd 

rows =[]
for nb ,ok in results :
    exp =EXPECTED .get (nb ,[])
    for path ,min_bytes in exp :
        exists =os .path .exists (path )
        size =os .path .getsize (path )if exists else 0 
        rows .append ({'notebook':nb ,'artifact':path ,'exists':exists ,'size_bytes':size ,'min_bytes':min_bytes ,'ok':exists and size >=min_bytes })

report =pd .DataFrame (rows )
report 


## 3) Resumen final y estado del pipeline

In [ ]:
if 'report'in globals ()and not report .empty :
    summary =report .groupby ('ok').size ().to_dict ()
    print ('✔️ Artefactos OK:',summary .get (True ,0 ),' | ❌ Fallos:',summary .get (False ,0 ))

    if summary .get (False ,0 )>0 :
        display (report [~report ['ok']])
else :
    print ('No hay artefactos en el reporte (¿se ejecutaron los notebooks?).')


### Notas
- Si ejecutas en Windows/Anaconda, asegúrate de abrir este notebook en el mismo entorno donde están las dependencias (ver `environment.yml`).
- Si un notebook depende de otro (por ejemplo, 04 depende de 02), ejecútalo en el orden indicado.
- Puedes añadir 00/01/01b a `NOTEBOOKS` si quieres correr absolutamente todo.